In [ ]:
# ===================== IMPORTS =====================
from fastapi import FastAPI, Depends, HTTPException, UploadFile, File, Form
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from sqlalchemy import create_engine, Column, Integer, String, DateTime
from sqlalchemy.orm import sessionmaker, declarative_base, Session
from datetime import datetime, timedelta
from jose import JWTError, jwt
from passlib.context import CryptContext
import shutil, os

# RAG
# ✅ FINAL WORKING IMPORTS

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ===================== APP =====================
app = FastAPI()

SECRET_KEY = "secret123"
ALGORITHM = "HS256"

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/login")
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

# ===================== DB =====================
DATABASE_URL = "sqlite:///./rag.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()

# ===================== MODELS =====================
class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True)
    email = Column(String, unique=True)
    password = Column(String)
    role = Column(String, default="Client")

class Document(Base):
    __tablename__ = "documents"
    id = Column(Integer, primary_key=True)
    title = Column(String)
    company_name = Column(String)
    document_type = Column(String)
    file_path = Column(String)
    uploaded_by = Column(String)
    created_at = Column(DateTime, default=datetime.utcnow)

Base.metadata.create_all(bind=engine)

# ===================== DEPENDENCY =====================
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

# ===================== AUTH =====================
def hash_password(password):
    return pwd_context.hash(password)

def verify_password(plain, hashed):
    return pwd_context.verify(plain, hashed)

def create_token(data: dict):
    return jwt.encode(data, SECRET_KEY, algorithm=ALGORITHM)

def get_current_user(token: str = Depends(oauth2_scheme), db: Session = Depends(get_db)):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        email = payload.get("sub")
    except:
        raise HTTPException(status_code=401, detail="Invalid token")

    user = db.query(User).filter(User.email == email).first()
    return user

# ===================== AUTH ROUTES =====================
@app.post("/auth/register")
def register(email: str, password: str, role: str = "Client", db: Session = Depends(get_db)):
    user = User(email=email, password=hash_password(password), role=role)
    db.add(user)
    db.commit()
    return {"msg": "User created"}

@app.post("/auth/login")
def login(form_data: OAuth2PasswordRequestForm = Depends(), db: Session = Depends(get_db)):
    user = db.query(User).filter(User.email == form_data.username).first()

    if not user or not verify_password(form_data.password, user.password):
        raise HTTPException(status_code=400, detail="Invalid credentials")

    token = create_token({"sub": user.email})
    return {"access_token": token, "token_type": "bearer"}

# ===================== UPLOAD =====================
UPLOAD_DIR = "uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

@app.post("/documents/upload")
def upload_doc(
    title: str = Form(...),
    company_name: str = Form(...),
    document_type: str = Form(...),
    file: UploadFile = File(...),
    user: User = Depends(get_current_user),
    db: Session = Depends(get_db)
):
    if user.role not in ["Admin", "Analyst"]:
        raise HTTPException(status_code=403, detail="Not allowed")

    file_path = os.path.join(UPLOAD_DIR, file.filename)

    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    doc = Document(
        title=title,
        company_name=company_name,
        document_type=document_type,
        file_path=file_path,
        uploaded_by=user.email
    )

    db.add(doc)
    db.commit()
    db.refresh(doc)

    return {"msg": "Uploaded", "doc_id": doc.id}

@app.delete("/documents/{doc_id}")
def delete_document(
    doc_id: int,
    user: User = Depends(get_current_user),
    db: Session = Depends(get_db)
):
    # 🔐 Only Admin can delete
    if user.role != "Admin":
        raise HTTPException(status_code=403, detail="Only Admin can delete documents")

    doc = db.query(Document).filter(Document.id == doc_id).first()

    if not doc:
        raise HTTPException(status_code=404, detail="Document not found")

    # 🗑️ Delete file from storage
    if os.path.exists(doc.file_path):
        os.remove(doc.file_path)

    # 🗑️ Delete FAISS index if exists
    index_path = f"faiss_index_{doc.id}"
    if os.path.exists(index_path):
        shutil.rmtree(index_path)

    # 🗑️ Delete from DB
    db.delete(doc)
    db.commit()

    return {"msg": "Document deleted successfully"}

# ===================== GET DOCS =====================
@app.get("/documents")
def get_docs(user: User = Depends(get_current_user), db: Session = Depends(get_db)):
    return db.query(Document).all()

# ===================== SEARCH =====================
@app.get("/documents/search")
def search_docs(company_name: str, user: User = Depends(get_current_user), db: Session = Depends(get_db)):
    return db.query(Document).filter(Document.company_name.contains(company_name)).all()

# ===================== INDEX =====================
@app.post("/rag/index-document/{doc_id}")
def index_doc(doc_id: int, db: Session = Depends(get_db)):
    doc = db.query(Document).filter(Document.id == doc_id).first()

    if not doc:
        raise HTTPException(status_code=404, detail="Document not found")

    if not os.path.exists(doc.file_path):
        raise HTTPException(status_code=400, detail="File not found")

    try:
        loader = PyPDFLoader(doc.file_path)
        pages = loader.load()

        splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        docs = splitter.split_documents(pages)

        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

        texts = [d.page_content for d in docs]

        vectorstore = FAISS.from_texts(texts, embeddings)

        vectorstore.save_local(f"faiss_index_{doc_id}")

        return {"msg": "Indexed successfully"}

    except Exception as e:
        return {"error": str(e)}

# ===================== RAG SEARCH =====================


# ===================== RUN =====================
import nest_asyncio
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config(app, host="0.0.0.0", port=8001)
server = uvicorn.Server(config)

await server.serve()
#http://127.0.0.1:8001/docs to start on the browser

INFO:     Started server process [18596]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


INFO:     127.0.0.1:60880 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:60880 - "GET /openapi.json HTTP/1.1" 200 OK


(trapped) error reading bcrypt version
Traceback (most recent call last):
  File "C:\Users\amins\anaconda3\Lib\site-packages\passlib\handlers\bcrypt.py", line 620, in _load_backend_mixin
    version = _bcrypt.__about__.__version__
              ^^^^^^^^^^^^^^^^^
AttributeError: module 'bcrypt' has no attribute '__about__'


INFO:     127.0.0.1:62895 - "POST /auth/login HTTP/1.1" 200 OK


C:\Users\amins\AppData\Local\Temp\ipykernel_18596\747226597.py:203: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


INFO:     127.0.0.1:64583 - "POST /rag/index-document/1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:63719 - "GET /documents/search?company_name=abc HTTP/1.1" 200 OK
INFO:     127.0.0.1:51459 - "DELETE /documents/1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:53185 - "GET /documents HTTP/1.1" 200 OK
